# 🥊 RNN vs LSTM vs GRU: The Vanishing Gradient Battle

We will build a simple sequence forecasting task and observe how Vanilla RNNs fail on long sequences compared to LSTMs and GRUs.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### The Toy Problem: Sine Wave Extrapolation

In [ ]:
def create_sine_data(seq_len=50, num_samples=1000):
    time_steps = np.linspace(0, 100, num_samples + seq_len)
    data = np.sin(time_steps)
    
    X, y = [], []
    for i in range(num_samples):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
        
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

seq_len = 20
X, y = create_sine_data(seq_len)
print(f"X shape: {X.shape}, y shape: {y.shape}")

### Defining the Architectures

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

### Training Loop (Exercise)
> Try training these three models on the same data. Notice how as you increase `seq_len` to 100 or 200, the Vanilla RNN stops converging while the LSTM and GRU continue to learn effectively!